# Notebook 04 â€” LDA Topic Modeling
## Crisis Alert System Â· Gensim LDA on Disaster Tweets

**Goal:** Find the optimal number of topics via coherence score, train the final LDA model, label topics as crisis/normal, and validate the 0.0â€“1.0 crisis score output for the ensemble.

In [1]:
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
import sys
sys.path.insert(0, str(ROOT))


import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 130, "axes.spines.top": False, "axes.spines.right": False})
CRISIS_COLOR = "#e63946"
NORMAL_COLOR = "#457b9d"

df = pd.read_csv("../data/processed/tweets_clean.csv").dropna(subset=["text_clean", "label"])
texts  = df["text_clean"].tolist()
labels = df["label"].astype(int).tolist()
print(f"Dataset: {len(df):,} rows")
print(df.head(3))

Dataset: 7,613 rows
   id                                         text_clean  \
0   1  our deeds are the reason of this earthquake ma...   
1   4             forest fire near la ronge sask. canada   
2   5  all residents asked to 'shelter in place' are ...   

                                       text_original  created_at  hour_bucket  \
0  Our Deeds are the Reason of this #earthquake M...         NaN          NaN   
1             Forest fire near La Ronge Sask. Canada         NaN          NaN   
2  All residents asked to 'shelter in place' are ...         NaN          NaN   

   label  engagement  
0      1           0  
1      1           0  
2      1           0  


## 1. Coherence Search â€” Find Optimal k

In [2]:
from src.models.lda_analyzer import LDAAnalyzer

lda_search = LDAAnalyzer()
coherence_df = lda_search.coherence_search(texts, k_range=range(5, 21))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(coherence_df["k"], coherence_df["coherence"], marker="o", color=CRISIS_COLOR, lw=2)
best_k = coherence_df.loc[coherence_df["coherence"].idxmax(), "k"]
best_c = coherence_df["coherence"].max()
ax.axvline(best_k, color="black", linestyle="--", lw=1.2, label=f"best k={best_k} (c={best_c:.4f})")
ax.set_xlabel("Number of Topics (k)")
ax.set_ylabel("Coherence Score (c_v)")
ax.set_title("LDA Coherence vs Number of Topics")
ax.legend()
plt.tight_layout()
plt.savefig("../outputs/charts/09_lda_coherence.png", bbox_inches="tight")
plt.show()
print(f"\nBest k = {best_k}  (coherence = {best_c:.4f})")
print(coherence_df.to_string(index=False))

  k= 5  coherence=0.5882
  k= 6  coherence=0.5661
  k= 7  coherence=0.5412
  k= 8  coherence=0.5634
  k= 9  coherence=0.5410
  k=10  coherence=0.5804
  k=11  coherence=0.5426
  k=12  coherence=0.5595
  k=13  coherence=0.5596
  k=14  coherence=0.5428
  k=15  coherence=0.5783
  k=16  coherence=0.5542
  k=17  coherence=0.5820
  k=18  coherence=0.5766
  k=19  coherence=0.5743
  k=20  coherence=0.5738

Best k = 5  (coherence = 0.5882)
 k  coherence
 5   0.588173
 6   0.566090
 7   0.541169
 8   0.563425
 9   0.540995
10   0.580371
11   0.542570
12   0.559483
13   0.559597
14   0.542817
15   0.578299
16   0.554222
17   0.581963
18   0.576556
19   0.574334
20   0.573815


C:\Users\Nafis\AppData\Local\Temp\ipykernel_21360\1345167558.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Train Final LDA Model

In [3]:
lda = LDAAnalyzer(n_topics=int(best_k), passes=15)
lda.fit(texts, labels)
lda.save("../outputs/models/lda_v1")

Training LDA: 5 topics, 15 passes, vocab=2,092
Crisis topics identified: [1, 2, 3]

Top words per topic  [C=crisis, N=normal]
  [N] Topic  0: new, emergency, may, video, full, old, army, bomb
  [C] Topic  1: fire, building, warning, severe, hostage, family, burning, volcano
  [C] Topic  2: man, family, weapon, thunderstorm, best, wreckage, malaysia, crash
  [C] Topic  3: still, storm, year, suicide, people, police, train, killed
  [N] Topic  4: disaster, back, california, see, fire, flood, going, death
LDA model saved to ../outputs/models/lda_v1


## 3. Topic Word Bars â€” Crisis vs Normal

In [4]:
topic_words = lda.topic_words(n_words=8)
n_topics    = lda.n_topics
crisis_set  = set(lda._crisis_topics)

cols = 4
rows = (n_topics + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(16, rows * 3))
axes = axes.flatten()

for tid in range(n_topics):
    words, probs = zip(*topic_words[tid])
    color = CRISIS_COLOR if tid in crisis_set else NORMAL_COLOR
    tag   = "CRISIS" if tid in crisis_set else "normal"
    axes[tid].barh(list(words)[::-1], list(probs)[::-1], color=color, alpha=0.85)
    axes[tid].set_title(f"Topic {tid} [{tag}]", fontsize=9, fontweight="bold")
    axes[tid].tick_params(axis="y", labelsize=8)

for ax in axes[n_topics:]:
    ax.set_visible(False)

plt.suptitle("LDA Topics â€” Top 8 Words (red=crisis, blue=normal)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../outputs/charts/10_lda_topics.png", bbox_inches="tight")
plt.show()

C:\Users\Nafis\AppData\Local\Temp\ipykernel_21360\1475259801.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Score Distribution vs Ground Truth

In [5]:
from sklearn.metrics import roc_auc_score, classification_report

scores     = lda.predict(texts)
scores_arr = np.array(scores)
labels_arr = np.array(labels)

# Threshold at median of scores (LDA scores are not calibrated like BERT)
threshold  = np.median(scores_arr)
preds      = (scores_arr >= threshold).astype(int)

print(f"LDA score range: [{scores_arr.min():.3f}, {scores_arr.max():.3f}]  median={threshold:.3f}")
print(f"ROC-AUC: {roc_auc_score(labels_arr, scores_arr):.3f}")
print()
print(classification_report(labels_arr, preds, target_names=["Normal","Crisis"]))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Score distribution
axes[0].hist(scores_arr[labels_arr==0], bins=40, alpha=0.7, color=NORMAL_COLOR, label="Normal", density=True)
axes[0].hist(scores_arr[labels_arr==1], bins=40, alpha=0.7, color=CRISIS_COLOR, label="Crisis", density=True)
axes[0].axvline(threshold, color="black", linestyle="--", lw=1.2, label=f"threshold={threshold:.2f}")
axes[0].set_xlabel("LDA crisis score")
axes[0].set_ylabel("Density")
axes[0].set_title("LDA Score Distribution by Label")
axes[0].legend()

# Box plot
data_to_plot = [scores_arr[labels_arr==0], scores_arr[labels_arr==1]]
bp = axes[1].boxplot(data_to_plot, labels=["Normal","Crisis"],
                     patch_artist=True, notch=True)
bp["boxes"][0].set_facecolor(NORMAL_COLOR)
bp["boxes"][1].set_facecolor(CRISIS_COLOR)
axes[1].set_ylabel("LDA crisis score")
axes[1].set_title("LDA Score Boxplot by Label")

plt.suptitle("LDA Analyzer â€” Score Validation", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../outputs/charts/11_lda_scores.png", bbox_inches="tight")
plt.show()

LDA score range: [0.020, 0.320]  median=0.213
ROC-AUC: 0.605

              precision    recall  f1-score   support

      Normal       0.64      0.56      0.60      4342
      Crisis       0.50      0.58      0.54      3271

    accuracy                           0.57      7613
   macro avg       0.57      0.57      0.57      7613
weighted avg       0.58      0.57      0.57      7613



C:\Users\Nafis\AppData\Local\Temp\ipykernel_21360\4283503911.py:29: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = axes[1].boxplot(data_to_plot, labels=["Normal","Crisis"],
C:\Users\Nafis\AppData\Local\Temp\ipykernel_21360\4283503911.py:39: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Save Full LDA Scores for Ensemble

In [ ]:
df["lda_score"] = scores
df[["id","lda_score"]].to_csv("../data/processed/tweets_lda_scores.csv", index=False)
print(f"Saved {len(df):,} LDA scores to data/processed/tweets_lda_scores.csv")
print(df[["id","lda_score"]].describe().round(3))

Saved 7,613 LDA scores to data/processed/tweets_lda_scores.csv
              id  lda_score
count   7613.000   7613.000
mean    5441.935      0.199
std     3137.116      0.082
min        1.000      0.020
25%     2734.000      0.133
50%     5408.000      0.213
75%     8146.000      0.272
max    10873.000      0.320


: 